# Funk SVD — Item Vector Training
Train Funk SVD on MovieLens 32M. Goal: produce a reliable item embedding matrix to use for fold-in and as content branch targets.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from surprise import SVD, Dataset, Reader

data_path = Path('../data')

## Load and filter MovieLens ratings

In [ ]:
MIN_RATINGS = 20

ratings = pd.read_csv(
    data_path / 'ml-32m' / 'ratings.csv',
    usecols=['userId', 'movieId', 'rating'],
)

counts = ratings['movieId'].value_counts()
valid_movies = counts[counts >= MIN_RATINGS].index
ratings = ratings[ratings['movieId'].isin(valid_movies)]

print(f"Movies with >= {MIN_RATINGS} ratings: {len(valid_movies):,}")
print(f"Ratings after filtering:              {len(ratings):,}")

## Train SVD

In [ ]:
N_FACTORS = 20  # k — keep < 35 (number of personal ratings for fold-in)
N_EPOCHS  = 40  # tuned via train/test RMSE analysis — optimal at reg=0.1
LR        = 0.005
REG       = 0.1  # tuned via reg sweep on 500k sample — best test RMSE at 0.9133

reader   = Reader(rating_scale=(0.5, 5.0))
dataset  = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
trainset = dataset.build_full_trainset()

algo = SVD(n_factors=N_FACTORS, n_epochs=N_EPOCHS, lr_all=LR, reg_all=REG, verbose=True)
algo.fit(trainset)

## Inspect item vectors

In [ ]:
print("Item matrix shape:", algo.qi.shape)  # (n_items, k)
print("Sample vector (first item):", algo.qi[0])

## Cross-validate to check hyperparameters

In [ ]:
from surprise.model_selection import cross_validate

# Sample to keep CV fast — 500k ratings is enough to evaluate hyperparameters
sample = ratings.sample(n=500_000, random_state=42)
cv_dataset = Dataset.load_from_df(sample[['userId', 'movieId', 'rating']], reader)

results = cross_validate(
    SVD(n_factors=20, n_epochs=20, lr_all=LR, reg_all=REG),
    cv_dataset,
    measures=['RMSE'],
    cv=3,
    verbose=True,
)

print(f"\nMean RMSE: {results['test_rmse'].mean():.4f} (+/- {results['test_rmse'].std():.4f})")

Evaluating RMSE of algorithm SVD on 3 split(s).

                  Fold 1  Fold 2  Fold 3  Mean    Std     
RMSE (testset)    0.9123  0.9109  0.9096  0.9109  0.0011  
Fit time          6.25    6.47    6.38    6.37    0.09    
Test time         6.41    5.18    7.45    6.35    0.93    

Mean RMSE: 0.9109 (+/- 0.0011)


## Learning curve — find convergence epoch

In [ ]:
# Small sample — only used to find the convergence point, not to evaluate absolute RMSE
lc_sample = ratings.sample(n=80_000, random_state=42)
lc_dataset = Dataset.load_from_df(lc_sample[['userId', 'movieId', 'rating']], reader)

epoch_counts = [10, 20, 30, 40, 50, 60]
rmse_scores = []

for n in epoch_counts:
    r = cross_validate(
        SVD(n_factors=N_FACTORS, n_epochs=n, lr_all=LR, reg_all=REG),
        lc_dataset,
        measures=['RMSE'],
        cv=2,
        verbose=False,
    )
    mean_rmse = r['test_rmse'].mean()
    rmse_scores.append(mean_rmse)
    print(f"n_epochs={n:3d}  RMSE={mean_rmse:.4f}")

for i in range(1, len(rmse_scores)):
    if rmse_scores[i - 1] - rmse_scores[i] < 0.001:
        print(f"\nConverged at n_epochs={epoch_counts[i]} (improvement < 0.001)")
        break

## Train vs test RMSE — overfitting check

In [ ]:
epoch_counts = [5, 10, 15, 20, 25, 30, 40, 50]
train_rmses = []
test_rmses  = []

for n in epoch_counts:
    r = cross_validate(
        SVD(n_factors=N_FACTORS, n_epochs=n, lr_all=LR, reg_all=REG),
        lc_dataset,  # 80k sample
        measures=['RMSE'],
        cv=2,
        return_train_measures=True,
        verbose=False,
    )
    train_rmses.append(r['train_rmse'].mean())
    test_rmses.append(r['test_rmse'].mean())
    print(f"n_epochs={n:3d}  train={train_rmses[-1]:.4f}  test={test_rmses[-1]:.4f}  gap={test_rmses[-1]-train_rmses[-1]:.4f}")

## Regularization comparison — reg=0.02 vs reg=0.05 (k=20)

In [ ]:
epoch_counts = [5, 10, 15, 20, 25, 30, 40, 50]

for reg in [0.02, 0.05, 0.08, 0.1]:
    print(f"\n--- reg={reg} ---")
    print(f"{'n_epochs':>10}  {'train':>7}  {'test':>7}  {'gap':>7}")
    for n in epoch_counts:
        r = cross_validate(
            SVD(n_factors=N_FACTORS, n_epochs=n, lr_all=LR, reg_all=reg),
            cv_dataset,  # 500k sample
            measures=['RMSE'],
            cv=2,
            return_train_measures=True,
            verbose=False,
        )
        train = r['train_rmse'].mean()
        test  = r['test_rmse'].mean()
        print(f"{n:>10}  {train:.4f}  {test:.4f}  {test-train:.4f}")

## Accuracy ceiling check — k=100, 30 epochs (not usable for fold-in)

In [11]:
r = cross_validate(
    SVD(n_factors=100, n_epochs=30, lr_all=0.001, reg_all=0.08),
    cv_dataset,  # 500k sample
    measures=['RMSE'],
    cv=3,
    verbose=True,
)
print(f"\nk=100, 30 epochs, reg=0.08  RMSE={r['test_rmse'].mean():.4f} (+/- {r['test_rmse'].std():.4f})")

Evaluating RMSE of algorithm SVD on 3 split(s).

                  Fold 1  Fold 2  Fold 3  Mean    Std     
RMSE (testset)    0.9554  0.9559  0.9595  0.9570  0.0018  
Fit time          10.03   12.32   12.97   11.77   1.26    
Test time         13.24   12.09   12.96   12.76   0.49    

k=100, 30 epochs, reg=0.08  RMSE=0.9570 (+/- 0.0018)


## Check matched personal films are present

In [ ]:
matched = pd.read_csv(data_path / 'movielens_matched.csv')

trained_movie_ids = set(int(trainset.to_raw_iid(i)) for i in range(trainset.n_items))

matched['in_trainset'] = matched['movieId'].isin(trained_movie_ids)
print(f"Personal films in trainset: {matched['in_trainset'].sum()} / {len(matched)}")
matched[['letterboxd_title', 'movieId', 'rating', 'in_trainset']]

## Save item vectors

In [ ]:
movieids    = np.array([int(trainset.to_raw_iid(i)) for i in range(trainset.n_items)])
item_biases = algo.bi  # same ordering as qi and movieids
global_mean = algo.trainset.global_mean

out_path = data_path / 'svd_item_vectors.npz'
np.savez(
    out_path,
    item_vectors=algo.qi,
    item_biases=item_biases,
    movieids=movieids,
    global_mean=np.array([global_mean]),
)

print(f"Saved {algo.qi.shape} item matrix to {out_path}")
print()
print("Load with:")
print("  d = np.load('data/svd_item_vectors.npz')")
print("  item_vectors = d['item_vectors']          # (n_items, k)")
print("  item_biases  = d['item_biases']            # (n_items,)")
print("  global_mean  = d['global_mean'][0]         # scalar")
print("  movieid_to_idx = {mid: i for i, mid in enumerate(d['movieids'])}")